# Part teòrica

**Variables:** Caselles consecutives amb direcció vertical o horitzontal de longitud > 1

**Domini:** Paraules del diccionari

**Restriccions:**
- Files i columnes han d'estar dins del taulell
- Files i columnes > 1 casella
- Interseccions de files i columnes tenen la mateixa lletra
- Si es troba una # acaba la fila o la columna
- Les paraules han d'estar escrites de dalt a baix i d'esquerra a dreta
- No es pot repetir una paraula
- Les paraules han de tenir una llargada menor o igual al màxim de m o n del taulell

**Tamany espai de solucions inicial:** D^v on *D* es el domini de paraules del diccionari i *v* les variables

**Estratègia de millora:** 


# Codi

## Comú

### Carregar biblioteques

In [97]:
import numpy as np
import timeit
import copy

### Loading data

In [98]:
def loadCrossword(file):
    data = []
    with open(file, 'r') as file:
        for line in file.readlines():
            elements = line.strip().split()
            data.append(elements)
    return(np.array(data))

def loadDictionary(file):
    words = {}
    with open(file, 'r') as file:
        for word in file.readlines():   
            if len(word.strip()) not in words.keys():
                words[len(word.strip())] = [word.strip()]
            else:
                v = words[len(word.strip())]
                v.append(word.strip())

    return words


In [99]:
def cercaVariablesHoritzontal(taulell, variables):
    for n, i in enumerate(taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

### Cerca de variables
```variable = [[pos_inicial], len, v/h]; vertical = 1, horitzontal = 0 ```

In [100]:
def cercaVariablesVertical(taulell, variables):
    transposed_taulell = taulell.transpose()
    for n, i in enumerate(transposed_taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [101]:
def cercaVariables(taulell, variables):
    cercaVariablesHoritzontal(taulell, variables)
    cercaVariablesVertical(taulell, variables)

In [102]:
def calculaPosicionsVariable(variable):
    variable_positions = []
    for x in range(variable[1]):
        i, j = 0, 0
        if variable[2] == 0: i = x
        else: j = x
        variable_positions.append([variable[0][0]+j, variable[0][1]+i])
    
    return variable_positions

### Letter of intersection

In [103]:
def intersectLetter(variable, word, assigned_variable, pos):
    if variable[2] == 0:
        return assigned_variable[1][pos[0] - assigned_variable[0][0][0]] == word[pos[1] - variable[0][1]]
    elif variable[2] == 1:
        return assigned_variable[1][pos[1] - assigned_variable[0][0][1]] == word[pos[0] - variable[0][0]]
    else: return False

### Comprovació de restriccions

In [104]:
def isValid(word, variable, assigned_variables):

    for assigned_variable in assigned_variables:    
        if word == assigned_variable[1]:
            return False
        
    for var in assigned_variables:
        for intersection in variable[3]:
            if intersection in var[0][3] and not intersectLetter(variable, word, var, intersection): return False
    return True

## Exercici 1

### Backtracking

```
Funcio Backtracking(LVA,LVNA,R,D)
    Si (LVNA és buida) llavors Retornar(LVA) fSi
    Var=Cap(LVNA);
    Per a cada (valor del Domini(Var, D) que podem assignar a Var) fer
        Si (SatisfaRestriccions([Var valor],LVA,R)) llavors
            Res=Backtracking(Insertar([Var, valor],LVA),Cua(LVNA),R,D);
            Si (Res és una solució completa) llavors
                Retornar(Res);
            Fsi
        Fsi
    Fper
    Retornar(Falla)
FFuncio

```

In [105]:
def backtracking(assigned_variables, variables, dictionary):
    if not variables:
        return assigned_variables

    var = variables[0]
    for word in dictionary[var[1]]:
        if isValid(word, var, assigned_variables):
            assigned_variables.append([var, word])
            result = backtracking(assigned_variables, variables[1:], dictionary)

            if result:
                return result
 
            assigned_variables.pop()  

    return None

## Exercici 2

### Inicialitzar dominis FC

In [106]:
def initDomains(variables, dictionary):
    fcDomains = {}
    for i, var in enumerate(variables):
        fcDomains[i] = list(dictionary[var[1]])
    return fcDomains

### ActualitzarDominis

In [107]:
def actualitzarDominis(variables, var, word, fcDomains, index, assigned_variables, originalDomains):
    for i in range(1, len(variables)):
        if variables[i] not in [assigned[0] for assigned in assigned_variables]:
            for position in variables[i][3]:
                if position in var[3]:
                    auxDomain = fcDomains[i + index][:]
                    for possible_word in fcDomains[i + index]:
                        if not intersectLetter(variables[i], possible_word, [var, word], position):
                            auxDomain.remove(possible_word)
                    if not auxDomain:
                        for i in range(1, len(variables)):
                            fcDomains[i + index] = originalDomains[i + index][:]
                        return None
                    
                    fcDomains[i + index] = auxDomain[:]

    return True

#### Calculate MRV

In [108]:
def calculateMRV(variables, assigned_variables, fcDomains):
    unassigned_variables = [var for var in variables if var not in [assigned[0] for assigned in assigned_variables]]
    mrv_values = [(var, len(fcDomains[i])) for i, var in enumerate(unassigned_variables)]
    mrv_values.sort(key=lambda x: x[1])
    return [var[0] for var in mrv_values]

### BackForwardChecking

In [109]:
def forwardChecking(assigned_variables, variables, dictionary, fcDomains, originalDomains):
    if not variables:
        return assigned_variables

    mrv_variables = calculateMRV(variables, assigned_variables, fcDomains)
    
    for var in mrv_variables:
        for word in dictionary[var[1]]:
            if isValid(word, var, assigned_variables) and actualitzarDominis(variables, var, word, fcDomains, len(assigned_variables), assigned_variables, originalDomains):
                assigned_variables.append([var, word])
                result = forwardChecking(assigned_variables, variables[1:], dictionary, fcDomains, originalDomains)

                if result:
                    return result

                assigned_variables.pop()

    return None

### Trobar interseccions

In [110]:
def trobaInterseccions(variables):
    posicions = []
    interseccions = []
    
    for variable in variables:
        for pos in calculaPosicionsVariable(variable):
            if pos not in posicions: posicions.append(pos)
            if pos in posicions: interseccions.append(pos)
            
    for variable in variables:
        aux_interseccions = []
        for pos in calculaPosicionsVariable(variable):
            if pos in interseccions: aux_interseccions.append(pos)
        variable.append(aux_interseccions)

### Print taulell final

In [111]:
def printSolution(assigned_variables, taulell):
    for variable, word in assigned_variables:
        positions = calculaPosicionsVariable(variable)
        for i, j in positions:
            taulell[i][j] = word[i - variable[0][0] if variable[2] == 1 else j - variable[0][1]]

    for row in taulell:
        print(" ".join(row))

### Main

In [112]:
if __name__ == '__main__':
    #Configuració general
    taulell = loadCrossword('crossword_CB_v3.txt')
    dictionary = loadDictionary('diccionari_A.txt')
    """
    #Backtracking----------------------------------------------------------------------------------------------------------------------
    assigned_variables = []
    variables = []
    cercaVariables(taulell, variables)
    trobaInterseccions(variables)
    
    backtracking = backtracking(assigned_variables, variables, dictionary)

    print("Resultat backtracking: ", backtracking)
    
    printSolution(assigned_variables, taulell)
    """
    #Forward Checking-------------------------------------------------------------------------------------------------------------------
    assigned_variables = []
    variables = []
    cercaVariables(taulell, variables)
    trobaInterseccions(variables)
    variables = sorted(variables, key=lambda x: len(x[3]))
    fcDomains = initDomains(variables, dictionary)
    originalDomains = initDomains(variables, dictionary)
    
    #variables = sorted(variables, key=lambda x: len(x[3]), reverse=True)
    
    FC = forwardChecking(assigned_variables, variables, dictionary, fcDomains, originalDomains)
    
    print("\n\nResultat FC:", FC)
    printSolution(assigned_variables, taulell)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\ppugi\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\ppugi\AppData\Local\Temp\ipykernel_25596\3218485370.py", line 29, in <module>
    FC = forwardChecking(assigned_variables, variables, dictionary, fcDomains, originalDomains)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ppugi\AppData\Local\Temp\ipykernel_25596\1731022087.py", line 11, in forwardChecking
    result = forwardChecking(assigned_variables, variables[1:], dictionary, fcDomains, originalDomains)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ppugi\AppData\Local\Temp\ipykernel_25596\1731022087.py", line 11, in forwardChecking
    result = forwardChecking(assigned_variables, variables[1:], dictionary, fcDomain